# RAKSHAK — Colab Hyperparameter Tuning (XGBoost + LightGBM)

Runs `RandomizedSearchCV` for XGBoost and LightGBM on Colab, per CLAUDE.md's hardware rule
(heavy training off the local M1). Local `train_model.py` already produced working, evaluated
baseline models with hand-picked defaults - this notebook searches for actually-good
hyperparameters instead of guessed ones, then hands the results back for local reassembly.

**Before running this**: `data/processed/cicids_clean.parquet` AND `models/selected_features.json`
must already be uploaded to `MyDrive/RAKSHAK/` in your Google Drive, and this notebook's runtime
must have Drive mounted (`drive.mount('/content/drive')`).

**Self-contained by design**: rather than cloning the GitHub repo (extra auth complexity for
uncertain benefit), this notebook re-implements the same split/scaling/SMOTE logic as
`train_model.py`, using the same constants (`RANDOM_STATE=42`, `TEST_SIZE=0.2`, etc.) - since
everything here is deterministic given the same input file, the resulting split is identical to
the local one. Feature *selection* is the one exception - see Section 4.

**Search runs on a 300K-row stratified subsample, not the full 2.1M-row training set** - a
standard technique to keep `RandomizedSearchCV` tractable within Colab's free-tier session
limits. The winning hyperparameters get refit on the *full* training set afterward for the
actual final model - search cheap, commit expensive.

**Known issue with this whole search-then-refit approach, found the hard way**: SMOTE (Section
5) is applied once, to the entire training set, before the search subsample is drawn - meaning
`RandomizedSearchCV`'s internal cross-validation folds are all built from already-synthetic data.
Since SMOTE-generated points are interpolated from real points that get scattered across both
sides of each fold's train/validation split, the search's own CV score is inflated - it measures
how well a model memorizes a dense synthetic neighborhood, not how well it generalizes. This
caused LightGBM's naive tuned model to collapse on real validation data despite a 0.9865 CV
score. The fix used here: after each search, don't trust `best_params_` blindly - refit a small
set of *manually chosen*, more conservative candidates directly on the full training set and
pick whichever actually performs best on real, untouched `X_val` (see the cell after Section 9).

**XGBoost gets saved unwrapped** (the raw model + its label encoder, as two separate files) -
wrapping it in the `LabelDecodingClassifier` class here would tag that class under Colab's
`__main__` module, causing the exact same load failure already hit and fixed locally. The
wrapping happens locally after download instead, using the real, properly-imported class.
LightGBM doesn't have this problem (handles string labels natively) and gets saved directly.

## 1. Mount Drive and verify the uploaded data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_PARQUET_PATH = Path('/content/drive/MyDrive/RAKSHAK/cicids_clean.parquet')
assert DRIVE_PARQUET_PATH.exists(), (
    f"Not found: {DRIVE_PARQUET_PATH} - check that cicids_clean.parquet was uploaded to "
    "MyDrive/RAKSHAK/ before running this notebook."
)
print(f"Found {DRIVE_PARQUET_PATH} ({DRIVE_PARQUET_PATH.stat().st_size / 1e6:.1f} MB)")

## 2. Install packages

Colab has most of this preinstalled already, but `imbalanced-learn` (for SMOTE) usually isn't -
`-q` keeps the install output quiet.

In [ ]:
!pip install -q imbalanced-learn lightgbm xgboost

## 3. Reconstruct the identical train/val/test split

Same constants, same order of operations as `split_train_val_test()` in `train_model.py` - since
this is deterministic given the same input file, X_train/X_val/X_test come out identical to the
local run's.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_FRACTION_OF_HOLDOUT = 0.5
TARGET_COLUMN = "Label"

df = pd.read_parquet(DRIVE_PARQUET_PATH)
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_holdout, y_holdout, test_size=VAL_FRACTION_OF_HOLDOUT, stratify=y_holdout, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

## 4. Feature selection

**Loads the canonical `selected_features.json` instead of recomputing it.** This notebook
originally recomputed feature selection fresh (its own quick Random Forest on a 10% sample) to
avoid hardcoding feature names - reasonable in principle, but it broke in practice: Colab's
scikit-learn version (1.6.1) differs from the local machine's (1.9.0), and even with an identical
`random_state=42`, a different sklearn version can consume its internal random number generator
differently - so the "quick RF" picked a genuinely different top-25 feature list than the one the
local Random Forest was trained on. That caused a hard `feature_names mismatch` error when
rebuilding the ensemble locally, since XGBoost/LightGBM and Random Forest disagreed on which 25
columns even exist. Loading the same canonical file everyone else uses guarantees agreement.

In [ ]:
import json

with open('/content/drive/MyDrive/RAKSHAK/selected_features.json') as f:
    top_features = json.load(f)

print("Loaded canonical features:")
for feature in top_features:
    print(f"  {feature}")

X_train = X_train[top_features]
X_val = X_val[top_features]
X_test = X_test[top_features]

## 5. Scale and apply SMOTE

Same rules as local: scaler fit on train only, transform val/test; SMOTE brings R2L/U2R up to
Probe's size (not full parity with Normal), applied to training data only.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE

SMOTE_TARGET_CLASS = "Probe"

scaler = MinMaxScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

class_counts = y_train.value_counts()
target_count = class_counts[SMOTE_TARGET_CLASS]
sampling_strategy = {
    label: target_count for label, count in class_counts.items() if count < target_count
}
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=RANDOM_STATE)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", y_train.value_counts().to_dict())
print(f"Final X_train: {X_train.shape}")

## 6. XGBoost hyperparameter search

Search runs on a 300K-row stratified subsample of the SMOTE'd training set - full RandomizedSearchCV
on all 2.1M+ rows would be impractical within Colab's session limits. `scoring="f1_macro"` is a
deliberate choice, not the default: macro F1 weighs every class equally, so the search is guided
toward configs that also help U2R/R2L, not just whichever config maximizes accuracy on the huge
Normal/DoS classes. `cv=3` and `n_iter=20` (60 total fits) keep runtime reasonable - increase
either if you have time/patience for a more thorough search.

XGBoost's sklearn wrapper needs integer-encoded labels for `fit()` (unlike RandomForestClassifier
or LightGBM) - same issue hit locally, handled the same way.

**Reminder** (see intro): this search's CV score is measured on SMOTE-contaminated folds and
runs optimistic - useful as a starting point, not as ground truth. Section 7 evaluates the refit
on real validation data, which is the number that actually matters.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

SEARCH_SAMPLE_SIZE = 300_000
N_ITER = 20
CV_FOLDS = 3

X_search, _, y_search, _ = train_test_split(
    X_train, y_train, train_size=SEARCH_SAMPLE_SIZE, stratify=y_train, random_state=RANDOM_STATE
)

xgb_label_encoder = LabelEncoder()
y_search_encoded = xgb_label_encoder.fit_transform(y_search)

xgb_param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.15, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
    param_distributions=xgb_param_dist,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    verbose=2,
)
xgb_search.fit(X_search, y_search_encoded)

print("Best XGBoost params:", xgb_search.best_params_)
print("Best XGBoost CV f1_macro:", xgb_search.best_score_)

## 7. Refit XGBoost on the full training set, evaluate on validation

The search only ever saw the 300K subsample; the actual final model is trained on all of
`X_train`. Evaluated on validation here (not test) - same discipline as the local threshold
tuning, test stays untouched until one final check after everything is decided.

In [ ]:
from sklearn.metrics import classification_report

y_train_encoded = xgb_label_encoder.transform(y_train)

final_xgb = XGBClassifier(tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, **xgb_search.best_params_)
final_xgb.fit(X_train, y_train_encoded)

y_val_pred = xgb_label_encoder.inverse_transform(final_xgb.predict(X_val))
print("=== Tuned XGBoost - validation ===")
print(classification_report(y_val, y_val_pred, digits=4))

## 8. LightGBM hyperparameter search

Same subsample-search-then-full-refit approach as XGBoost. No label encoding needed here -
LightGBM's sklearn wrapper accepts the original string labels directly.

**Same CV-optimism caveat as Section 6 applies here too** - and hits LightGBM harder in practice.
Its leaf-wise, less-regularized-by-default growth (especially with unbounded `max_depth=-1` in
the search space) is more prone to memorizing the dense synthetic SMOTE clusters than XGBoost's
growth strategy, so trusting this cell's `best_params_` directly produced a model that collapsed
on real validation data (macro F1 ~0.28) despite a 0.9865 CV score. Kept here anyway for the
before/after comparison - the actual final model comes from the manually-regularized cell after
Section 9, not from `lgbm_search.best_params_`.

In [ ]:
from lightgbm import LGBMClassifier

lgbm_param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 7, 8, -1],
    "learning_rate": [0.01, 0.05, 0.1, 0.15, 0.2],
    "num_leaves": [15, 31, 63, 127],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
}

lgbm_search = RandomizedSearchCV(
    LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    param_distributions=lgbm_param_dist,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    verbose=2,
)
lgbm_search.fit(X_search, y_search)

print("Best LightGBM params:", lgbm_search.best_params_)
print("Best LightGBM CV f1_macro:", lgbm_search.best_score_)

## 9. Refit LightGBM on the full training set, evaluate on validation

Expected to look bad (see Section 8's caveat) - this cell is kept to show the actual before/after
comparison rather than silently skipping the broken result. The next cell fixes it.

In [ ]:
final_lgbm = LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1, **lgbm_search.best_params_)
final_lgbm.fit(X_train, y_train)

y_val_pred = final_lgbm.predict(X_val)
print("=== Tuned LightGBM - validation ===")
print(classification_report(y_val, y_val_pred, digits=4))

## 9b. Fix: manually-regularized LightGBM candidates, selected on real validation data

Instead of trusting `lgbm_search.best_params_` (chosen using the leaky, SMOTE-contaminated CV
score from Section 8), this tries a small number of deliberately more conservative candidates -
capped `max_depth`, added `min_child_samples`/`reg_alpha`/`reg_lambda` - and picks whichever one
actually scores best on real, untouched `X_val`. No leakage possible here: `X_val` was never
SMOTE'd and never touched during search, so this is a genuine, honest comparison.

In practice, the winning candidate turned out to have `reg_alpha=0.0, reg_lambda=0.0` - what
actually fixed it was capping `max_depth`/`n_estimators` away from the search's unbounded
`max_depth=-1, n_estimators=400`, not the explicit L1/L2 penalty terms.

In [ ]:
from sklearn.metrics import f1_score, classification_report

candidate_configs = [
    dict(n_estimators=200, max_depth=6, learning_rate=0.1,  num_leaves=31, min_child_samples=50,  reg_alpha=0.0, reg_lambda=0.0),
    dict(n_estimators=200, max_depth=6, learning_rate=0.1,  num_leaves=31, min_child_samples=100, reg_alpha=0.1, reg_lambda=0.1),
    dict(n_estimators=300, max_depth=6, learning_rate=0.05, num_leaves=31, min_child_samples=100, reg_alpha=0.1, reg_lambda=0.1),
    dict(n_estimators=200, max_depth=4, learning_rate=0.1,  num_leaves=15, min_child_samples=200, reg_alpha=1.0, reg_lambda=1.0),
    dict(n_estimators=300, max_depth=8, learning_rate=0.05, num_leaves=63, min_child_samples=200, reg_alpha=1.0, reg_lambda=1.0),
    dict(n_estimators=200, max_depth=6, learning_rate=0.1,  num_leaves=31, min_child_samples=300, reg_alpha=1.0, reg_lambda=1.0),
]

results = []
for i, params in enumerate(candidate_configs):
    model = LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1, **params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    macro_f1 = f1_score(y_val, preds, average="macro")
    print(f"Candidate {i}: {params}")
    print(f"  macro F1 on real X_val = {macro_f1:.4f}\n")
    results.append((macro_f1, params, model))

results.sort(key=lambda r: r[0], reverse=True)
best_f1, best_params, best_lgbm = results[0]
print("=== Best candidate ===")
print(best_params)
print(f"macro F1 = {best_f1:.4f}")

final_lgbm = best_lgbm  # overwrite so Section 10 saves this one, not the leaky search's result
y_val_pred = final_lgbm.predict(X_val)
print("\n=== Best regularized LightGBM - validation ===")
print(classification_report(y_val, y_val_pred, digits=4))

## 10. Save tuned models to Drive

XGBoost saved as two separate files (raw model + label encoder) rather than wrapped in
`LabelDecodingClassifier` - see this notebook's intro for why. LightGBM saves directly.

In [ ]:
import joblib

OUTPUT_DIR = Path('/content/drive/MyDrive/RAKSHAK/tuned_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_xgb, OUTPUT_DIR / 'xgb_model_raw.joblib')
joblib.dump(xgb_label_encoder, OUTPUT_DIR / 'xgb_label_encoder.joblib')
joblib.dump(final_lgbm, OUTPUT_DIR / 'lgbm_model.joblib')

print(f"Saved to {OUTPUT_DIR}:")
for f in OUTPUT_DIR.iterdir():
    print(f"  {f.name}")

## Next steps (back on your local machine)

1. Download `xgb_model_raw.joblib`, `xgb_label_encoder.joblib`, and `lgbm_model.joblib` from
   `MyDrive/RAKSHAK/tuned_models/` into `models/tuned_from_colab/` in the local project.
2. Run `python src/reassemble_tuned_models.py` - wraps the raw XGBoost model with the real,
   properly-imported `LabelDecodingClassifier` class and replaces `models/xgb_model.joblib` and
   `models/lgbm_model.joblib`.
3. Run `python src/rebuild_ensemble.py` - rebuilds the soft-voting ensemble from the current
   three models and re-sweeps the U2R decision threshold on validation (the old
   `U2R_DECISION_THRESHOLD=0.80` was tuned for the old, untuned ensemble and isn't assumed to
   still be optimal).